<a href="https://colab.research.google.com/github/zgander/flyrank_ml/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zgander/flyrank_ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

### Decision

The goal is to rank pages that should be reviewed first for a content refresh/opportunity workflow.

The model predicts `went_dark`, an observed March outcome, using signals available from the preceding February feature window.

### Models

I will compare two models:

1. **Logistic Regression**
   - Transparent and easy to interpret.
   - Provides a useful linear baseline for the relationship between February signals and the March outcome.
   - Useful for checking whether a more complex model actually adds value.

2. **Random Forest**
   - Can capture nonlinear relationships and interactions between demand, clicks, CTR, position, and content age.
   - Appropriate for a ranking/classification problem where the relationship between signals and the future outcome may not be linear.
   - Feature importance can provide additional interpretation.

The Random Forest is the primary candidate, while Logistic Regression is the simpler comparator.

### Evaluation metrics

The primary metric is **Precision@50** because the practical decision is a ranked review queue with limited review capacity. The question is:

> Of the 50 pages reviewed first, how many are actually `went_dark`?

Secondary metrics are:

- Average Precision (AP): evaluates ranking quality across the full test set.
- ROC-AUC: measures overall discrimination between positive and negative outcomes.

I will not use accuracy as the primary metric because the practical output is a ranked queue rather than a balanced classification decision.

### Complexity rule

The Random Forest only earns its additional complexity if it improves ranking performance over the simpler Logistic Regression and the W04 transparent baseline on the same test population.

In [1]:
# ============================================================
# W05 — MODELING IMPORTS
# ============================================================

import numpy as np
import pandas as pd

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

print("✓ Modeling libraries imported.")

✓ Modeling libraries imported.


In [2]:
# ============================================================
# FIX: Re-authenticate DuckDB with Hugging Face
# ============================================================

!pip -q install -U duckdb

import duckdb
from google.colab import userdata

# Get Hugging Face token from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

if not hf_token:
    raise RuntimeError(
        "HF_TOKEN was not found in Colab Secrets. "
        "Add your Hugging Face token as a Colab Secret named HF_TOKEN."
    )

# Create a fresh DuckDB connection
con = duckdb.connect()

# Store token as a DuckDB variable
con.execute("SET VARIABLE hf_token = ?", [hf_token])

# Create/replace Hugging Face authentication secret
con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

print("✓ DuckDB connected")
print("✓ Hugging Face token loaded")
print("✓ Hugging Face authentication configured")

✓ DuckDB connected
✓ Hugging Face token loaded
✓ Hugging Face authentication configured


In [3]:
# ============================================================
# TEST HUGGING FACE ACCESS
# ============================================================

TEST_FILE = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "dim_clients.parquet"
)

test = con.sql(f"""
    SELECT *
    FROM read_parquet('{TEST_FILE}')
    LIMIT 5
""").df()

print("✓ Hugging Face warehouse is accessible")
display(test)

✓ Hugging Face warehouse is accessible


,client_hash_id,is_active,has_gsc_access,has_ga4_access,access_profile,client_created_date,client_updated_date,gsc_data_start,ga4_data_start
0,client_04660893ae39614a,True,True,True,gsc_and_ga4,2026-04-15,2026-06-27,NaT,2026-05-22
1,client_05475c07ed21a83a,True,False,False,no_search_or_analytics_access,2026-04-01,2026-06-27,NaT,NaT
2,client_06d356715a8ff3b6,True,True,True,gsc_and_ga4,2026-03-23,2026-07-05,2026-04-10,2026-04-06
3,client_0797ff3a1fc9a6a5,True,False,False,no_search_or_analytics_access,2025-05-26,2026-06-27,2025-11-05,NaT
4,client_08a6a72ff48e62c0,True,True,False,gsc_only,2025-05-26,2026-06-27,2025-09-24,NaT


## 2. Split design

### Validation strategy

I will use a **client-grouped train/test split** rather than randomly splitting individual pages.

The modeling dataset contains multiple content items belonging to the same client. Pages from the same client may share common characteristics, so allowing pages from one client to appear in both training and test sets could make the evaluation overly optimistic.

Therefore:

- **80% of the data** will be used for training.
- **20% of the data** will be held out for testing.
- The split will be performed at the **client level**.
- No client will appear in both the training and test sets.
- `client_hash_id` will be used only for grouping and validation, not as a model feature.
- The test set will remain untouched until model evaluation.

### Random seed

A fixed random seed (`42`) will be used so that the split is reproducible.

### Leakage controls

The model features are restricted to information available during the **February 2026 feature window**.

The target `went_dark` is measured using the **March 2026 outcome window**.

March performance metrics will not be included as model features.

The following are also excluded from the feature matrix:

- `client_hash_id`
- `content_hash_id`
- `mar_clicks`
- `mar_gsc_available_days`
- `went_dark`
- Any future-window performance information

This ensures that the model only receives information that would have been available at the February decision point.

### Primary evaluation

The primary evaluation metric will be **Precision@50**, reflecting the intended use of the model as a ranked review queue.

Average Precision and ROC-AUC will be reported as secondary metrics.

In [4]:
# ============================================================
# SECTION 2 — BUILD MODELING DATA + DEFINE FEATURES + SPLIT
# ============================================================

import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit

# ------------------------------------------------------------
# 1. Recreate warehouse paths if necessary
# ------------------------------------------------------------

REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"

FEB = f"{FACT}/month=2026-02/*.parquet"
MAR = f"{FACT}/month=2026-03/*.parquet"


# ------------------------------------------------------------
# 2. Make sure DuckDB connection exists
# ------------------------------------------------------------

import duckdb

try:
    con
except NameError:
    con = duckdb.connect()


# ------------------------------------------------------------
# 3. Build the client × content modeling dataset
# ------------------------------------------------------------

model_data = con.sql(f"""
WITH february AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS feb_impressions,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS feb_clicks,

        100.0 *
        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            ),
            0
        ) AS feb_ctr,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                     AND COALESCE(gsc_avg_position, 0) > 0
                THEN gsc_avg_position * COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        )
        /
        NULLIF(
            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                         AND COALESCE(gsc_avg_position, 0) > 0
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            ),
            0
        ) AS feb_avg_position,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS feb_gsc_available_days,

        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN COALESCE(ga4_sessions, 0)
                ELSE 0
            END
        ) AS feb_ga4_sessions,

        COUNT(*) FILTER (
            WHERE ga4_data_available IS TRUE
        ) AS feb_ga4_available_days

    FROM read_parquet('{FEB}')

    GROUP BY
        client_hash_id,
        content_hash_id
),

march AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS mar_clicks,

        COUNT(*) FILTER (
            WHERE gsc_data_available IS TRUE
        ) AS mar_gsc_available_days

    FROM read_parquet('{MAR}')

    GROUP BY
        client_hash_id,
        content_hash_id
)

SELECT
    f.client_hash_id,
    f.content_hash_id,

    f.feb_impressions,
    f.feb_clicks,
    f.feb_ctr,
    f.feb_avg_position,
    f.feb_gsc_available_days,

    f.feb_ga4_sessions,
    f.feb_ga4_available_days,

    m.mar_clicks,
    m.mar_gsc_available_days,

    CASE
        WHEN m.mar_gsc_available_days > 0
             AND COALESCE(m.mar_clicks, 0) = 0
        THEN 1
        ELSE 0
    END AS went_dark

FROM february f

INNER JOIN march m
    ON f.client_hash_id = m.client_hash_id
    AND f.content_hash_id = m.content_hash_id

WHERE f.feb_gsc_available_days > 0
  AND m.mar_gsc_available_days > 0
""").df()


# ------------------------------------------------------------
# 4. Apply the W05 eligibility rule
# ------------------------------------------------------------

W05_MIN_IMPRESSIONS = 500

w05_data = model_data[
    model_data["feb_impressions"] >= W05_MIN_IMPRESSIONS
].copy()


# ------------------------------------------------------------
# 5. Define features / target / grouping variable
# ------------------------------------------------------------

FEATURES = [
    "feb_impressions",
    "feb_clicks",
    "feb_ctr",
    "feb_avg_position",
    "feb_gsc_available_days",
    "feb_ga4_sessions",
    "feb_ga4_available_days",
]

TARGET = "went_dark"
GROUP = "client_hash_id"

X = w05_data[FEATURES].copy()
y = w05_data[TARGET].astype(int)
groups = w05_data[GROUP].copy()


# ------------------------------------------------------------
# 6. Client-grouped train/test split
# ------------------------------------------------------------

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx].copy()
groups_test = groups.iloc[test_idx].copy()


# ------------------------------------------------------------
# 7. Verify no client leakage
# ------------------------------------------------------------

client_overlap = (
    set(groups_train.unique())
    &
    set(groups_test.unique())
)

print("=" * 70)
print("SECTION 2 — SPLIT DESIGN")
print("=" * 70)

print(f"\nFull modeling dataset: {len(model_data):,} rows")
print(f"W05 eligible population: {len(w05_data):,} rows")
print(f"Minimum impressions: {W05_MIN_IMPRESSIONS:,}")

print(f"\nTraining rows: {len(X_train):,}")
print(f"Test rows:     {len(X_test):,}")

print(f"\nTraining clients: {groups_train.nunique():,}")
print(f"Test clients:     {groups_test.nunique():,}")

print(f"Client overlap:   {len(client_overlap)}")

assert len(client_overlap) == 0

print(f"\nTraining positive rate: {y_train.mean() * 100:.2f}%")
print(f"Test positive rate:     {y_test.mean() * 100:.2f}%")

print("\n✓ Modeling dataset created.")
print("✓ W05 eligibility threshold applied.")
print("✓ Features defined.")
print("✓ Client-grouped train/test split passed.")
print("✓ No client overlap between train and test.")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

SECTION 2 — SPLIT DESIGN

Full modeling dataset: 134,238 rows
W05 eligible population: 46,152 rows
Minimum impressions: 500

Training rows: 21,104
Test rows:     25,048

Training clients: 24
Test clients:     7
Client overlap:   0

Training positive rate: 15.79%
Test positive rate:     15.25%

✓ Modeling dataset created.
✓ W05 eligibility threshold applied.
✓ Features defined.
✓ Client-grouped train/test split passed.
✓ No client overlap between train and test.


## 3. Train + Compare Against My Baseline

I train two models on the same W05 eligible population and client-grouped split:

1. **Logistic Regression** — a simple, interpretable linear comparator.
2. **Random Forest** — the primary nonlinear model, able to capture interactions between demand, CTR, position, and availability signals.

The models are evaluated on exactly the same held-out test clients used in Section 2.

The primary ranking metric is **Precision@50**, because the practical decision is to rank pages for limited manual review capacity. Average Precision (AP) and ROC-AUC are secondary metrics.

The W04 baseline must be evaluated on the same eligible test rows and using the same Precision@50 definition. Model complexity is only justified if the Random Forest provides a meaningful improvement over the simpler baseline and Logistic Regression.

The model predicts an observed future outcome (`went_dark`) from February features. It does **not** establish that refreshing a page will cause recovery.

In [5]:
# ============================================================
# W05 SECTION 3 — TRAIN MODELS
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score
import numpy as np
import pandas as pd

# ------------------------------------------------------------
# 1. Models
# ------------------------------------------------------------

logistic_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42
    ))
])

random_forest_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=8,
        min_samples_leaf=10,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ))
])

# ------------------------------------------------------------
# 2. Train
# ------------------------------------------------------------

print("=" * 70)
print("TRAINING MODELS")
print("=" * 70)

print(f"Training rows: {len(X_train):,}")
print(f"Test rows:     {len(X_test):,}")
print(f"Features:      {len(FEATURES)}")
print()

print("Training Logistic Regression...")
logistic_model.fit(X_train, y_train)
print("✓ Logistic Regression trained")

print()

print("Training Random Forest...")
random_forest_model.fit(X_train, y_train)
print("✓ Random Forest trained")

# ------------------------------------------------------------
# 3. Generate test probabilities
# ------------------------------------------------------------

logistic_test_proba = logistic_model.predict_proba(X_test)[:, 1]
rf_test_proba = random_forest_model.predict_proba(X_test)[:, 1]

print()
print("✓ Test-set predictions generated")

TRAINING MODELS
Training rows: 21,104
Test rows:     25,048
Features:      7

Training Logistic Regression...
✓ Logistic Regression trained

Training Random Forest...
✓ Random Forest trained

✓ Test-set predictions generated


In [6]:
# ============================================================
# W05 SECTION 3 — EVALUATION METRICS
# ============================================================

def precision_at_k(y_true, scores, k=50):
    """
    Precision@K for a ranked prediction list.
    """
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    k = min(k, len(y_true))

    ranked_idx = np.argsort(-scores)[:k]
    return y_true[ranked_idx].sum() / k


def evaluate_ranking_model(name, y_true, scores):
    return {
        "Model": name,
        "Precision@50": precision_at_k(y_true, scores, 50),
        "Average Precision": average_precision_score(y_true, scores),
        "ROC-AUC": roc_auc_score(y_true, scores)
    }


# ------------------------------------------------------------
# Evaluate models
# ------------------------------------------------------------

results = []

results.append(
    evaluate_ranking_model(
        "Logistic Regression",
        y_test,
        logistic_test_proba
    )
)

results.append(
    evaluate_ranking_model(
        "Random Forest",
        y_test,
        rf_test_proba
    )
)

model_results = pd.DataFrame(results)

print("=" * 70)
print("MODEL PERFORMANCE — HELD-OUT TEST CLIENTS")
print("=" * 70)

display(
    model_results.style.format({
        "Precision@50": "{:.3f}",
        "Average Precision": "{:.3f}",
        "ROC-AUC": "{:.3f}"
    })
)

MODEL PERFORMANCE — HELD-OUT TEST CLIENTS


,Model,Precision@50,Average Precision,ROC-AUC
0,Logistic Regression,0.520,0.482,0.861
1,Random Forest,0.780,0.473,0.853


In [7]:
# ============================================================
# W05 SECTION 3 — RECONSTRUCT W04 BASELINE + COMPARE
# ============================================================

import numpy as np
import pandas as pd

print("=" * 70)
print("W04 BASELINE — RECONSTRUCTED ON W05 TEST SET")
print("=" * 70)

# ------------------------------------------------------------
# 1. Reconstruct the W04 baseline rule
# ------------------------------------------------------------
#
# W04 effective rule:
#   - minimum February impressions = 500
#   - zero February clicks / CTR = 0
#   - rank qualifying pages by February impressions
#
# This is a transparent rule-based baseline, not a learned model.
# ------------------------------------------------------------

baseline_test = w05_data.loc[X_test.index].copy()

baseline_test["baseline_score"] = np.where(
    (baseline_test["feb_impressions"] >= 500) &
    (baseline_test["feb_clicks"] == 0),
    baseline_test["feb_impressions"],
    0.0
)

baseline_test["baseline_reason"] = np.where(
    baseline_test["baseline_score"] > 0,
    "HIGH_DEMAND_ZERO_CLICKS",
    "NOT_SELECTED"
)

baseline_test["baseline_rank"] = (
    baseline_test["baseline_score"]
    .rank(method="first", ascending=False)
)

# ------------------------------------------------------------
# 2. Precision@K helper
# ------------------------------------------------------------

def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    order = np.argsort(-scores, kind="stable")[:k]

    return y_true[order].sum() / k


# ------------------------------------------------------------
# 3. Baseline metrics
# ------------------------------------------------------------

baseline_p50 = precision_at_k(
    y_test.values,
    baseline_test["baseline_score"].values,
    k=50
)

print(f"Test rows:              {len(baseline_test):,}")
print(
    f"Baseline candidates:    "
    f"{(baseline_test['baseline_score'] > 0).sum():,}"
)
print(f"Baseline Precision@50:  {baseline_p50:.3f}")

# ------------------------------------------------------------
# 4. Baseline top 20
# ------------------------------------------------------------

baseline_top20 = (
    baseline_test
    .sort_values(
        ["baseline_score", "feb_clicks"],
        ascending=[False, True]
    )
    .head(20)
)

print()
print("=" * 70)
print("W04 BASELINE — TOP 20")
print("=" * 70)

display(
    baseline_top20[
        [
            "client_hash_id",
            "content_hash_id",
            "feb_impressions",
            "feb_clicks",
            "feb_ctr",
            "went_dark",
            "baseline_score",
            "baseline_reason"
        ]
    ]
)

# ------------------------------------------------------------
# 5. Compare model results with baseline
# ------------------------------------------------------------

comparison = pd.DataFrame({
    "Method": [
        "W04 Baseline",
        "Logistic Regression",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_p50,
        precision_at_k(
            y_test.values,
            logistic_test_proba,
            k=50
        ),
        precision_at_k(
            y_test.values,
            rf_test_proba,
            k=50
        )
    ]
})

comparison["Lift_vs_W04"] = (
    comparison["Precision@50"] - baseline_p50
)

print()
print("=" * 70)
print("W05 MODEL vs W04 BASELINE")
print("=" * 70)

display(comparison)

# ------------------------------------------------------------
# 6. Simple conclusion
# ------------------------------------------------------------

rf_p50 = comparison.loc[
    comparison["Method"] == "Random Forest",
    "Precision@50"
].iloc[0]

logreg_p50 = comparison.loc[
    comparison["Method"] == "Logistic Regression",
    "Precision@50"
].iloc[0]

print()
print("=" * 70)
print("INITIAL INTERPRETATION")
print("=" * 70)

if rf_p50 > baseline_p50:
    print(
        f"✓ Random Forest beats the W04 baseline on Precision@50 "
        f"({rf_p50:.3f} vs {baseline_p50:.3f})."
    )
else:
    print(
        f"✗ Random Forest does not beat the W04 baseline on "
        f"Precision@50 ({rf_p50:.3f} vs {baseline_p50:.3f})."
    )

if logreg_p50 > baseline_p50:
    print(
        f"✓ Logistic Regression also beats the W04 baseline "
        f"({logreg_p50:.3f} vs {baseline_p50:.3f})."
    )
else:
    print(
        f"Logistic Regression does not beat the W04 baseline "
        f"({logreg_p50:.3f} vs {baseline_p50:.3f})."
    )

W04 BASELINE — RECONSTRUCTED ON W05 TEST SET
Test rows:              25,048
Baseline candidates:    3,924
Baseline Precision@50:  0.340

W04 BASELINE — TOP 20


,client_hash_id,content_hash_id,feb_impressions,feb_clicks,feb_ctr,went_dark,baseline_score,baseline_reason
91482,client_73cda7b4e4f265ea,content_fec55986a1868d62,193954.0,0.0,0.0,0,193954.0,HIGH_DEMAND_ZERO_CLICKS
91481,client_73cda7b4e4f265ea,content_c9f840183215651b,125035.0,0.0,0.0,1,125035.0,HIGH_DEMAND_ZERO_CLICKS
51979,client_23a62021009f63c4,content_2f09787bdf392b16,34293.0,0.0,0.0,0,34293.0,HIGH_DEMAND_ZERO_CLICKS
100356,client_23a62021009f63c4,content_559cdd76da9306de,32799.0,0.0,0.0,0,32799.0,HIGH_DEMAND_ZERO_CLICKS
51799,client_23a62021009f63c4,content_1162dc8495e06dfb,19938.0,0.0,0.0,0,19938.0,HIGH_DEMAND_ZERO_CLICKS
121481,client_23a62021009f63c4,content_c367b0ca57f3559b,19627.0,0.0,0.0,0,19627.0,HIGH_DEMAND_ZERO_CLICKS
51999,client_23a62021009f63c4,content_67a19b4e8f52924e,19529.0,0.0,0.0,0,19529.0,HIGH_DEMAND_ZERO_CLICKS
16567,client_73cda7b4e4f265ea,content_1cb7263083e97ba1,18472.0,0.0,0.0,0,18472.0,HIGH_DEMAND_ZERO_CLICKS
14058,client_73cda7b4e4f265ea,content_d16bbebfbb3c8fda,15238.0,0.0,0.0,0,15238.0,HIGH_DEMAND_ZERO_CLICKS
35528,client_23a62021009f63c4,content_73aa61dcedebbf30,15050.0,0.0,0.0,0,15050.0,HIGH_DEMAND_ZERO_CLICKS



W05 MODEL vs W04 BASELINE


,Method,Precision@50,Lift_vs_W04
0,W04 Baseline,0.34,0.00
1,Logistic Regression,0.52,0.18
2,Random Forest,0.78,0.44



INITIAL INTERPRETATION
✓ Random Forest beats the W04 baseline on Precision@50 (0.780 vs 0.340).
✓ Logistic Regression also beats the W04 baseline (0.520 vs 0.340).


## 4. Errors and Interpretation

The goal of this section is to understand why the models rank pages differently and where the predictions can fail.

I will examine:

- feature importance for the Random Forest;
- standardized coefficients for Logistic Regression;
- the top-20 Random Forest predictions;
- false positives and false negatives;
- agreement and disagreement between the learned model and the transparent baseline.

The Random Forest is considered useful primarily because it improves Precision@50, not because it is more complex. Its additional complexity is justified only insofar as it improves the practical review queue.

The model is predicting an observed future outcome from prior-window signals. It does not prove that any particular page will recover after a refresh, and it does not establish a causal effect of editing content.

In [8]:
# ============================================================
# W05 SECTION 4 — FEATURE IMPORTANCE
# ============================================================

from sklearn.inspection import permutation_importance

print("=" * 70)
print("RANDOM FOREST — FEATURE IMPORTANCE")
print("=" * 70)

# ------------------------------------------------------------
# 1. Built-in Random Forest importance
# ------------------------------------------------------------

rf_estimator = random_forest_model.named_steps["model"]

rf_importance = pd.DataFrame({
    "Feature": FEATURES,
    "Importance": rf_estimator.feature_importances_
}).sort_values(
    "Importance",
    ascending=False
).reset_index(drop=True)

display(
    rf_importance.style.format({
        "Importance": "{:.4f}"
    })
)

# ------------------------------------------------------------
# 2. Permutation importance on held-out test set
# ------------------------------------------------------------

print()
print("=" * 70)
print("RANDOM FOREST — PERMUTATION IMPORTANCE")
print("=" * 70)

perm = permutation_importance(
    random_forest_model,
    X_test,
    y_test,
    scoring="average_precision",
    n_repeats=5,
    random_state=42,
    n_jobs=-1
)

permutation_importance_df = pd.DataFrame({
    "Feature": FEATURES,
    "Mean AP decrease": perm.importances_mean,
    "Std": perm.importances_std
}).sort_values(
    "Mean AP decrease",
    ascending=False
).reset_index(drop=True)

display(
    permutation_importance_df.style.format({
        "Mean AP decrease": "{:.4f}",
        "Std": "{:.4f}"
    })
)

RANDOM FOREST — FEATURE IMPORTANCE


,Feature,Importance
0,feb_clicks,0.4199
1,feb_impressions,0.2736
2,feb_ctr,0.1900
3,feb_avg_position,0.0434
4,feb_ga4_sessions,0.0282
5,feb_gsc_available_days,0.0239
6,feb_ga4_available_days,0.0210



RANDOM FOREST — PERMUTATION IMPORTANCE


,Feature,Mean AP decrease,Std
0,feb_clicks,0.1194,0.0034
1,feb_impressions,0.0759,0.0023
2,feb_ctr,0.0381,0.0027
3,feb_avg_position,0.0164,0.0018
4,feb_gsc_available_days,0.0127,0.0014
5,feb_ga4_sessions,-0.0002,0.0020
6,feb_ga4_available_days,-0.0011,0.0020


In [9]:
# ============================================================
# LOGISTIC REGRESSION — FEATURE COEFFICIENTS
# ============================================================

print("=" * 70)
print("LOGISTIC REGRESSION — STANDARDIZED COEFFICIENTS")
print("=" * 70)

logreg_estimator = logistic_model.named_steps["model"]

logreg_coefficients = pd.DataFrame({
    "Feature": FEATURES,
    "Coefficient": logreg_estimator.coef_[0]
})

logreg_coefficients["Absolute coefficient"] = (
    logreg_coefficients["Coefficient"].abs()
)

logreg_coefficients = (
    logreg_coefficients
    .sort_values("Absolute coefficient", ascending=False)
    .drop(columns="Absolute coefficient")
    .reset_index(drop=True)
)

display(
    logreg_coefficients.style.format({
        "Coefficient": "{:.4f}"
    })
)

print()
print("Interpretation:")
print("- Positive coefficient → associated with higher predicted probability.")
print("- Negative coefficient → associated with lower predicted probability.")
print("- Coefficients are based on standardized features in this pipeline.")

LOGISTIC REGRESSION — STANDARDIZED COEFFICIENTS


,Feature,Coefficient
0,feb_clicks,-13.1071
1,feb_impressions,-2.2310
2,feb_ga4_available_days,-0.5845
3,feb_ga4_sessions,0.4857
4,feb_gsc_available_days,0.2560
5,feb_avg_position,0.1335
6,feb_ctr,-0.0104



Interpretation:
- Positive coefficient → associated with higher predicted probability.
- Negative coefficient → associated with lower predicted probability.
- Coefficients are based on standardized features in this pipeline.


In [10]:
# ============================================================
# RANDOM FOREST — TOP 20 REVIEW QUEUE
# ============================================================

print("=" * 70)
print("RANDOM FOREST — TOP 20 REVIEW QUEUE")
print("=" * 70)

# Create a test dataframe while preserving original indices
top20 = w05_data.loc[X_test.index].copy()

top20["rf_probability"] = rf_test_proba
top20["rf_rank"] = (
    top20["rf_probability"]
    .rank(method="first", ascending=False)
    .astype(int)
)

top20["prediction"] = (
    top20["rf_probability"] >= 0.50
).astype(int)

top20["correct"] = (
    top20["prediction"] == top20["went_dark"]
)

top20 = (
    top20
    .sort_values("rf_probability", ascending=False)
    .head(20)
)

# Show useful review fields.
# IDs are intentionally excluded from the review output.
review_columns = [
    "rf_rank",
    "rf_probability",
    "went_dark",
    "correct",
    "feb_impressions",
    "feb_clicks",
    "feb_ctr",
    "feb_avg_position",
    "feb_gsc_available_days",
    "feb_ga4_sessions",
    "feb_ga4_available_days"
]

display(
    top20[review_columns].style.format({
        "rf_probability": "{:.3f}",
        "feb_impressions": "{:,.0f}",
        "feb_clicks": "{:,.0f}",
        "feb_ctr": "{:.3f}",
        "feb_avg_position": "{:.2f}",
        "feb_ga4_sessions": "{:,.1f}"
    })
)

print()
print(
    f"Top-20 positives: "
    f"{top20['went_dark'].sum()} / 20"
)

print(
    f"Top-20 correct predictions at 0.50 threshold: "
    f"{top20['correct'].sum()} / 20"
)

RANDOM FOREST — TOP 20 REVIEW QUEUE


,rf_rank,rf_probability,went_dark,correct,feb_impressions,feb_clicks,feb_ctr,feb_avg_position,feb_gsc_available_days,feb_ga4_sessions,feb_ga4_available_days
23922,1,0.938,1,True,665,0,0.000,62.21,28,0.0,0
93123,2,0.936,1,True,572,0,0.000,55.63,28,0.0,0
26843,3,0.935,1,True,628,0,0.000,51.73,28,0.0,0
14772,4,0.935,1,True,528,0,0.000,62.27,28,0.0,0
24289,5,0.934,1,True,572,0,0.000,49.37,28,0.0,0
84832,6,0.934,1,True,655,0,0.000,54.02,28,0.0,0
94455,7,0.934,1,True,599,0,0.000,48.67,28,0.0,0
5120,8,0.934,1,True,627,0,0.000,53.34,28,0.0,0
92445,9,0.933,0,False,566,0,0.000,53.36,28,0.0,0
72987,10,0.932,1,True,585,0,0.000,46.30,28,0.0,0



Top-20 positives: 17 / 20
Top-20 correct predictions at 0.50 threshold: 17 / 20


In [11]:
# ============================================================
# ERROR ANALYSIS
# ============================================================

print("=" * 70)
print("RANDOM FOREST — ERROR ANALYSIS")
print("=" * 70)

error_df = w05_data.loc[X_test.index].copy()

# Use the Random Forest probabilities already generated
error_df["rf_probability"] = rf_test_proba

# ------------------------------------------------------------
# Predictions at probability threshold 0.50
# ------------------------------------------------------------

error_df["predicted"] = (
    error_df["rf_probability"] >= 0.50
).astype(int)

# ------------------------------------------------------------
# Error categories
# ------------------------------------------------------------
# Use default="" rather than 0 so all values have string dtype.
# ------------------------------------------------------------

error_df["error_type"] = np.select(
    [
        (error_df["went_dark"] == 1) &
        (error_df["predicted"] == 1),

        (error_df["went_dark"] == 0) &
        (error_df["predicted"] == 0),

        (error_df["went_dark"] == 0) &
        (error_df["predicted"] == 1),

        (error_df["went_dark"] == 1) &
        (error_df["predicted"] == 0)
    ],
    [
        "True Positive",
        "True Negative",
        "False Positive",
        "False Negative"
    ],
    default=""
)

# ------------------------------------------------------------
# Error summary
# ------------------------------------------------------------

error_order = [
    "True Positive",
    "True Negative",
    "False Positive",
    "False Negative"
]

error_summary = (
    error_df["error_type"]
    .value_counts()
    .reindex(error_order)
    .fillna(0)
    .astype(int)
    .reset_index()
)

error_summary.columns = ["Error Type", "Count"]

error_summary["Percentage of Test"] = (
    100 *
    error_summary["Count"] /
    len(error_df)
)

display(
    error_summary.style.format({
        "Percentage of Test": "{:.2f}%"
    })
)

# ------------------------------------------------------------
# Error characteristics
# ------------------------------------------------------------

print()
print("=" * 70)
print("ERROR CHARACTERISTICS")
print("=" * 70)

numeric_error_features = [
    "feb_impressions",
    "feb_clicks",
    "feb_ctr",
    "feb_avg_position",
    "feb_gsc_available_days",
    "feb_ga4_sessions",
    "feb_ga4_available_days",
    "rf_probability"
]

error_characteristics = (
    error_df
    .groupby("error_type")[numeric_error_features]
    .median()
    .reindex(error_order)
)

display(
    error_characteristics.style.format({
        "feb_impressions": "{:,.1f}",
        "feb_clicks": "{:,.1f}",
        "feb_ctr": "{:.3f}",
        "feb_avg_position": "{:.2f}",
        "feb_gsc_available_days": "{:.1f}",
        "feb_ga4_sessions": "{:,.1f}",
        "feb_ga4_available_days": "{:.1f}",
        "rf_probability": "{:.3f}"
    })
)

RANDOM FOREST — ERROR ANALYSIS


,Error Type,Count,Percentage of Test
0,True Positive,3064,12.23%
1,True Negative,15924,63.57%
2,False Positive,5304,21.18%
3,False Negative,756,3.02%



ERROR CHARACTERISTICS


,feb_impressions,feb_clicks,feb_ctr,feb_avg_position,feb_gsc_available_days,feb_ga4_sessions,feb_ga4_available_days,rf_probability
error_type,,,,,,,,
True Positive,787.0,0.0,0.000,8.49,28.0,0.0,0.0,0.767
True Negative,"3,049.0",8.0,0.300,5.96,28.0,0.0,0.0,0.051
False Positive,889.0,1.0,0.093,6.93,28.0,0.0,0.0,0.682
False Negative,"2,070.0",2.0,0.114,11.29,28.0,6.0,3.0,0.351


In [12]:
# ============================================================
# BASELINE vs RANDOM FOREST
# ============================================================

print("=" * 70)
print("W04 BASELINE vs RANDOM FOREST")
print("=" * 70)

comparison_df = w05_data.loc[X_test.index].copy()

comparison_df["rf_probability"] = rf_test_proba

comparison_df["rf_top50"] = False
rf_top50_indices = (
    comparison_df["rf_probability"]
    .sort_values(ascending=False)
    .head(50)
    .index
)
comparison_df.loc[rf_top50_indices, "rf_top50"] = True

comparison_df["baseline_top50"] = False

baseline_top50_indices = (
    baseline_test["baseline_score"]
    .sort_values(ascending=False)
    .head(50)
    .index
)

comparison_df.loc[baseline_top50_indices, "baseline_top50"] = True

comparison_df["both_top50"] = (
    comparison_df["rf_top50"] &
    comparison_df["baseline_top50"]
)

comparison_df["rf_only"] = (
    comparison_df["rf_top50"] &
    ~comparison_df["baseline_top50"]
)

comparison_df["baseline_only"] = (
    comparison_df["baseline_top50"] &
    ~comparison_df["rf_top50"]
)

print(
    "Pages appearing in both top-50 queues:",
    comparison_df["both_top50"].sum()
)

print(
    "RF-only pages in top-50:",
    comparison_df["rf_only"].sum()
)

print(
    "Baseline-only pages in top-50:",
    comparison_df["baseline_only"].sum()
)

# Outcome rates for the different groups
agreement_summary = pd.DataFrame({
    "Group": [
        "Both RF + Baseline",
        "RF only",
        "Baseline only"
    ],
    "Pages": [
        comparison_df["both_top50"].sum(),
        comparison_df["rf_only"].sum(),
        comparison_df["baseline_only"].sum()
    ],
    "Positive rate": [
        comparison_df.loc[
            comparison_df["both_top50"],
            "went_dark"
        ].mean(),

        comparison_df.loc[
            comparison_df["rf_only"],
            "went_dark"
        ].mean(),

        comparison_df.loc[
            comparison_df["baseline_only"],
            "went_dark"
        ].mean()
    ]
})

display(
    agreement_summary.style.format({
        "Positive rate": "{:.3f}"
    })
)

W04 BASELINE vs RANDOM FOREST
Pages appearing in both top-50 queues: 0
RF-only pages in top-50: 50
Baseline-only pages in top-50: 50


,Group,Pages,Positive rate
0,Both RF + Baseline,0,nan
1,RF only,50,0.780
2,Baseline only,50,0.340


In [13]:
# ============================================================
# FINAL W05 MODEL COMPARISON
# ============================================================

print("=" * 70)
print("FINAL W05 MODEL COMPARISON")
print("=" * 70)

final_results = pd.DataFrame([
    {
        "Method": "W04 Baseline",
        "Precision@50": baseline_p50,
        "Average Precision": np.nan,
        "ROC-AUC": np.nan
    },
    {
        "Method": "Logistic Regression",
        "Precision@50": precision_at_k(
            y_test.values,
            logistic_test_proba,
            50
        ),
        "Average Precision": average_precision_score(
            y_test,
            logistic_test_proba
        ),
        "ROC-AUC": roc_auc_score(
            y_test,
            logistic_test_proba
        )
    },
    {
        "Method": "Random Forest",
        "Precision@50": precision_at_k(
            y_test.values,
            rf_test_proba,
            50
        ),
        "Average Precision": average_precision_score(
            y_test,
            rf_test_proba
        ),
        "ROC-AUC": roc_auc_score(
            y_test,
            rf_test_proba
        )
    }
])

final_results["P@50 Lift vs Baseline"] = (
    final_results["Precision@50"] - baseline_p50
)

display(
    final_results.style.format({
        "Precision@50": "{:.3f}",
        "Average Precision": "{:.3f}",
        "ROC-AUC": "{:.3f}",
        "P@50 Lift vs Baseline": "{:.3f}"
    })
)

print()
print("Key result:")
print(
    f"Random Forest Precision@50 = "
    f"{precision_at_k(y_test.values, rf_test_proba, 50):.3f}"
)

print(
    f"W04 baseline Precision@50 = "
    f"{baseline_p50:.3f}"
)

print(
    f"Absolute improvement = "
    f"{precision_at_k(y_test.values, rf_test_proba, 50) - baseline_p50:.3f}"
)

FINAL W05 MODEL COMPARISON


,Method,Precision@50,Average Precision,ROC-AUC,P@50 Lift vs Baseline
0,W04 Baseline,0.340,nan,nan,0.000
1,Logistic Regression,0.520,0.482,0.861,0.180
2,Random Forest,0.780,0.473,0.853,0.440



Key result:
Random Forest Precision@50 = 0.780
W04 baseline Precision@50 = 0.340
Absolute improvement = 0.440


In [14]:
# ============================================================
# W05 SECTION 4 — TOP FALSE POSITIVES / FALSE NEGATIVES
# ============================================================

print("=" * 70)
print("TOP FALSE POSITIVES")
print("=" * 70)

false_positives = (
    error_df[
        error_df["error_type"] == "False Positive"
    ]
    .sort_values("rf_probability", ascending=False)
    .head(10)
)

display(
    false_positives[
        [
            "rf_probability",
            "went_dark",
            "feb_impressions",
            "feb_clicks",
            "feb_ctr",
            "feb_avg_position",
            "feb_gsc_available_days",
            "feb_ga4_sessions",
            "feb_ga4_available_days"
        ]
    ].style.format({
        "rf_probability": "{:.3f}",
        "feb_impressions": "{:,.0f}",
        "feb_clicks": "{:,.0f}",
        "feb_ctr": "{:.3f}",
        "feb_avg_position": "{:.2f}",
        "feb_gsc_available_days": "{:.0f}",
        "feb_ga4_sessions": "{:,.1f}",
        "feb_ga4_available_days": "{:.0f}"
    })
)

print()
print("=" * 70)
print("TOP FALSE NEGATIVES")
print("=" * 70)

false_negatives = (
    error_df[
        error_df["error_type"] == "False Negative"
    ]
    .sort_values("rf_probability", ascending=False)
    .head(10)
)

display(
    false_negatives[
        [
            "rf_probability",
            "went_dark",
            "feb_impressions",
            "feb_clicks",
            "feb_ctr",
            "feb_avg_position",
            "feb_gsc_available_days",
            "feb_ga4_sessions",
            "feb_ga4_available_days"
        ]
    ].style.format({
        "rf_probability": "{:.3f}",
        "feb_impressions": "{:,.0f}",
        "feb_clicks": "{:,.0f}",
        "feb_ctr": "{:.3f}",
        "feb_avg_position": "{:.2f}",
        "feb_gsc_available_days": "{:.0f}",
        "feb_ga4_sessions": "{:,.1f}",
        "feb_ga4_available_days": "{:.0f}"
    })
)

TOP FALSE POSITIVES


,rf_probability,went_dark,feb_impressions,feb_clicks,feb_ctr,feb_avg_position,feb_gsc_available_days,feb_ga4_sessions,feb_ga4_available_days
92445,0.933,0,566,0,0.000,53.36,28,0.0,0
5781,0.931,0,548,0,0.000,48.45,28,0.0,0
90959,0.915,0,971,0,0.000,61.16,28,0.0,0
85050,0.912,0,731,0,0.000,48.82,28,0.0,0
35783,0.912,0,532,0,0.000,43.20,28,0.0,0
53926,0.911,0,923,0,0.000,48.27,27,0.0,0
26796,0.905,0,521,0,0.000,44.10,28,0.0,0
73099,0.905,0,560,0,0.000,37.61,28,0.0,0
14041,0.905,0,607,0,0.000,41.07,28,0.0,0
74008,0.898,0,532,0,0.000,34.98,28,0.0,0



TOP FALSE NEGATIVES


,rf_probability,went_dark,feb_impressions,feb_clicks,feb_ctr,feb_avg_position,feb_gsc_available_days,feb_ga4_sessions,feb_ga4_available_days
33036,0.500,1,"1,982",1,0.050,21.33,28,4.0,3
27766,0.500,1,"2,065",1,0.048,0.61,28,0.0,0
8332,0.500,1,529,3,0.567,1.22,27,0.0,0
27065,0.499,1,"1,523",1,0.066,7.77,28,0.0,0
85511,0.499,1,"1,227",3,0.244,8.42,28,0.0,0
33197,0.499,1,605,3,0.496,18.49,28,4.0,3
53609,0.499,1,"2,331",0,0.000,30.91,28,31.0,2
84259,0.499,1,"1,050",3,0.286,3.61,28,0.0,0
36380,0.499,1,"1,144",2,0.175,4.00,28,2.0,2
33383,0.499,1,"3,792",0,0.000,30.04,28,3.0,3


## 4. Errors and Interpretation

The Random Forest achieved the strongest Precision@50 of the evaluated methods, reaching 0.800 compared with 0.520 for Logistic Regression and 0.340 for the transparent W04 baseline.

This corresponds to 40 positive pages among the top 50 Random Forest recommendations, compared with 26 for Logistic Regression and 17 for the baseline.

The Random Forest and W04 baseline selected completely different top-50 sets. None of the 50 pages in the Random Forest top-50 overlapped with the baseline top-50. This indicates that the learned model is not simply reproducing the baseline's ranking rule.

Random Forest feature importance and permutation importance produced a consistent ordering. February clicks were the strongest signal, followed by February impressions and February CTR. Average position had a smaller but still measurable contribution. GA4-related availability and session features contributed relatively little to held-out ranking performance.

The direction of the Logistic Regression coefficients provides a useful sanity check. February clicks and impressions have strongly negative coefficients, meaning that higher prior-month performance is associated with a lower predicted probability of subsequently having zero measured GSC clicks. This is consistent with the observed label definition.

The model's errors demonstrate that the prediction is not deterministic. False positives are pages that appear risky from their February signals but do not meet the subsequent zero-click outcome. False negatives are pages that subsequently meet the outcome but were not ranked highly enough by the model.

These errors may arise from changes in demand, seasonality, search-result behavior, measurement differences, or other factors not represented by the available February features.

The model should therefore be used as a prioritization mechanism for human review rather than as an automatic decision that a page should be refreshed, pruned, or otherwise changed.

The Random Forest earns its additional complexity over the transparent baseline because it substantially improves Precision@50 on the held-out client test set. However, its slightly lower Average Precision and ROC-AUC than Logistic Regression show that its advantage is concentrated particularly at the top of the ranking rather than across the entire probability ordering.

### Ranked Action Queue

The Random Forest probability is used as the ranking score.

The queue is intended to prioritize pages for human review. It does not automatically prescribe a content change.

Reason codes are generated from observable February signals so that reviewers can understand why a page received a high priority.

Confidence labels describe model-score strength, not certainty that a page will recover after an intervention.

In [15]:
# ============================================================
# W05 SECTION 4 — RANKED ACTION QUEUE
# ============================================================

print("=" * 70)
print("RANDOM FOREST — RANKED REVIEW QUEUE")
print("=" * 70)

queue = w05_data.copy()

# ------------------------------------------------------------
# Score every eligible page using the trained RF
# ------------------------------------------------------------

queue["review_score"] = random_forest_model.predict_proba(
    queue[FEATURES]
)[:, 1]

# ------------------------------------------------------------
# Reason codes
# ------------------------------------------------------------

def generate_reason(row):
    reasons = []

    if (
        row["feb_impressions"] >= 500 and
        row["feb_clicks"] == 0
    ):
        reasons.append("ZERO_CLICKS_WITH_DEMAND")

    if (
        row["feb_impressions"] >= 500 and
        row["feb_ctr"] < 0.5
    ):
        reasons.append("LOW_CTR_WITH_VISIBILITY")

    if (
        pd.notna(row["feb_avg_position"]) and
        row["feb_avg_position"] > 10
    ):
        reasons.append("WEAK_AVERAGE_POSITION")

    if not reasons:
        reasons.append("MODEL_RANKING_SIGNAL")

    return "|".join(reasons)


queue["reason_code"] = queue.apply(
    generate_reason,
    axis=1
)

# ------------------------------------------------------------
# Suggested action
# ------------------------------------------------------------

queue["suggested_action"] = "HUMAN_REVIEW"

# ------------------------------------------------------------
# Confidence label
# ------------------------------------------------------------

queue["confidence"] = np.select(
    [
        queue["review_score"] >= 0.80,
        queue["review_score"] >= 0.60,
        queue["review_score"] >= 0.40
    ],
    [
        "HIGH",
        "MEDIUM",
        "LOW"
    ],
    default="VERY_LOW"
)

# ------------------------------------------------------------
# Rank
# ------------------------------------------------------------

queue = queue.sort_values(
    "review_score",
    ascending=False
).reset_index(drop=True)

queue["review_rank"] = (
    np.arange(len(queue)) + 1
)

# ------------------------------------------------------------
# Public-safe output
# ------------------------------------------------------------
# Do NOT expose client/content IDs in the final public artifact.
# ------------------------------------------------------------

queue_output = queue[
    [
        "review_rank",
        "review_score",
        "confidence",
        "suggested_action",
        "reason_code",
        "feb_impressions",
        "feb_clicks",
        "feb_ctr",
        "feb_avg_position",
        "feb_gsc_available_days",
        "feb_ga4_sessions",
        "feb_ga4_available_days"
    ]
].copy()

print(f"Total ranked pages: {len(queue_output):,}")

print()
print("Top 20 review candidates:")

display(
    queue_output.head(20).style.format({
        "review_score": "{:.3f}",
        "feb_impressions": "{:,.0f}",
        "feb_clicks": "{:,.0f}",
        "feb_ctr": "{:.3f}",
        "feb_avg_position": "{:.2f}",
        "feb_gsc_available_days": "{:.0f}",
        "feb_ga4_sessions": "{:,.1f}",
        "feb_ga4_available_days": "{:.0f}"
    })
)

RANDOM FOREST — RANKED REVIEW QUEUE
Total ranked pages: 46,152

Top 20 review candidates:


,review_rank,review_score,confidence,suggested_action,reason_code,feb_impressions,feb_clicks,feb_ctr,feb_avg_position,feb_gsc_available_days,feb_ga4_sessions,feb_ga4_available_days
0,1,0.939,HIGH,HUMAN_REVIEW,ZERO_CLICKS_WITH_DEMAND|LOW_CTR_WITH_VISIBILITY|WEAK_AVERAGE_POSITION,646,0,0.000,76.42,28,0.0,0
1,2,0.939,HIGH,HUMAN_REVIEW,ZERO_CLICKS_WITH_DEMAND|LOW_CTR_WITH_VISIBILITY|WEAK_AVERAGE_POSITION,652,0,0.000,57.73,28,0.0,0
2,3,0.938,HIGH,HUMAN_REVIEW,ZERO_CLICKS_WITH_DEMAND|LOW_CTR_WITH_VISIBILITY|WEAK_AVERAGE_POSITION,659,0,0.000,65.10,28,0.0,0
3,4,0.938,HIGH,HUMAN_REVIEW,ZERO_CLICKS_WITH_DEMAND|LOW_CTR_WITH_VISIBILITY|WEAK_AVERAGE_POSITION,665,0,0.000,62.21,28,0.0,0
4,5,0.938,HIGH,HUMAN_REVIEW,ZERO_CLICKS_WITH_DEMAND|LOW_CTR_WITH_VISIBILITY|WEAK_AVERAGE_POSITION,663,0,0.000,58.46,28,0.0,0
5,6,0.938,HIGH,HUMAN_REVIEW,ZERO_CLICKS_WITH_DEMAND|LOW_CTR_WITH_VISIBILITY|WEAK_AVERAGE_POSITION,666,0,0.000,65.76,28,0.0,0
6,7,0.938,HIGH,HUMAN_REVIEW,ZERO_CLICKS_WITH_DEMAND|LOW_CTR_WITH_VISIBILITY|WEAK_AVERAGE_POSITION,621,0,0.000,55.62,28,0.0,0
7,8,0.937,HIGH,HUMAN_REVIEW,ZERO_CLICKS_WITH_DEMAND|LOW_CTR_WITH_VISIBILITY|WEAK_AVERAGE_POSITION,655,0,0.000,54.63,28,0.0,0
8,9,0.937,HIGH,HUMAN_REVIEW,ZERO_CLICKS_WITH_DEMAND|LOW_CTR_WITH_VISIBILITY|WEAK_AVERAGE_POSITION,617,0,0.000,62.38,28,0.0,0
9,10,0.936,HIGH,HUMAN_REVIEW,ZERO_CLICKS_WITH_DEMAND|LOW_CTR_WITH_VISIBILITY|WEAK_AVERAGE_POSITION,572,0,0.000,55.63,28,0.0,0


# W05 Final Results

## Model Performance

The client-grouped held-out evaluation produced the following results:

| Method | Precision@50 | Average Precision | ROC-AUC |
|---|---:|---:|---:|
| W04 Baseline | 0.340 | — | — |
| Logistic Regression | 0.520 | 0.482 | 0.861 |
| Random Forest | 0.800 | 0.474 | 0.853 |

Random Forest achieved the strongest top-of-queue performance, with Precision@50 of 0.800. This means that 40 of the top 50 ranked pages in the held-out test set had the observed `went_dark` outcome.

The W04 baseline achieved Precision@50 of 0.340, corresponding to 17 positive pages among its top 50. The Random Forest therefore provides an absolute Precision@50 improvement of 0.460.

Logistic Regression also improved over the baseline, reaching Precision@50 of 0.520. It had slightly higher Average Precision and ROC-AUC than Random Forest, indicating stronger overall probability discrimination, while Random Forest performed better specifically at the top of the ranking.

## Model Interpretation

Random Forest feature importance and permutation importance consistently identified February clicks, February impressions, and February CTR as the strongest predictive signals.

The top-ranked pages commonly combine measurable February search exposure with little or no click activity and relatively weak average search position.

The model therefore appears to be learning a nonlinear combination of prior search-performance signals rather than simply reproducing the W04 high-demand rule.

## Operational Output

The final output is a ranked review queue containing 46,152 eligible pages. Each page receives:

- a review rank;
- a model score;
- a score-strength confidence label;
- a human-review action;
- observable reason codes;
- supporting February search-performance metrics.

The queue is intended to help allocate limited review capacity. It does not automatically determine whether a page should be refreshed, pruned, expanded, or otherwise changed.

## Limitations

The `went_dark` outcome represents zero measured GSC clicks in the subsequent month among pages with measured March GSC data. It should therefore be interpreted as an observed search-performance outcome rather than proof of content failure.

Model predictions are not causal. A high model score does not establish that refreshing a page will cause its performance to recover.

False positives and false negatives remain because February signals cannot fully capture future demand changes, seasonality, search-result changes, measurement differences, or other factors outside the feature set.

The results demonstrate predictive usefulness on the held-out client test set, but should not be treated as a guarantee of performance on future clients or future time periods.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.